# Lab 1 — Build an MCP server (EP13)

The theory video opened with a `search_papers` tool that needs a new connector every time it moves to another program.
In this lab we turn it into an **MCP server** with FastMCP (Listing 21.1), then connect a client to it (Listing 21.2)

**What you'll do**
1. Write a paper-search server: two **tools** · one **resource** · one **prompt**
2. Connect a client over **stdio** — initialize, and negotiate capabilities
3. See how FastMCP builds the **inputSchema** from type hints and docstrings
4. Call a tool · read a resource · fetch a prompt · and see an error
5. Let the **model pick a tool** from the descriptions the server sends

The paper records are teaching data (the arXiv ids, titles and years are real); the model's tool choices come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §21.2–21.3, §21.5, §21.7.1–21.7.2, Listings 21.1–21.2 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs less than 0.01 US cents) |
| Packages | `mcp`, `openai`, `python-dotenv` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "mcp==1.30.0" "openai>=1.50" python-dotenv

## 1. Write the server: `paper_server.py`

The server runs as a **separate process** (stdio transport: the client starts the server as a child program),
so we write it to a file with `%%writefile` — the first cell creates the file, the next ones append with `-a`

| Part of the server | In MCP it's a | Who uses it |
|---|---|---|
| `search_papers`, `cite_paper` | **tool** | the model decides to call it |
| `paper://{paper_id}` | **resource** | the host puts it in the context |
| `summarize_paper` | **prompt** | the user picks it |

In [ ]:
%%writefile paper_server.py
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("cogito-papers")

PAPERS = {  # teaching data: real arXiv ids, titles and years; short summaries written for this lab
    "2211.14730": {"title": "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers",
                   "authors": "Nie et al.", "year": 2023, "summary": "PatchTST: split each series into patches, one channel at a time"},
    "2310.06625": {"title": "iTransformer: Inverted Transformers Are Effective for Time Series Forecasting",
                   "authors": "Liu et al.", "year": 2024, "summary": "attention across variables instead of across time steps"},
    "2310.10688": {"title": "A decoder-only foundation model for time-series forecasting",
                   "authors": "Das et al.", "year": 2024, "summary": "TimesFM: one pretrained model that forecasts many series zero-shot"},
    "2012.07436": {"title": "Informer: Beyond Efficient Transformer for Long Sequence Time-Series Forecasting",
                   "authors": "Zhou et al.", "year": 2021, "summary": "sparse attention to make long forecasts cheaper"},
    "2205.13504": {"title": "Are Transformers Effective for Time Series Forecasting?",
                   "authors": "Zeng et al.", "year": 2023, "summary": "a simple linear model beats several transformers"},
}

In [ ]:
%%writefile -a paper_server.py

@mcp.tool()
def search_papers(query: str, max_results: int = 3) -> str:
    """Search the paper library by keyword in the title or summary.
    Use this to find papers on a topic. Returns a JSON list of {id, title, year}, best match first."""
    words = query.lower().split()
    scored = []
    for pid, p in PAPERS.items():
        text = (p["title"] + " " + p["summary"]).lower()
        score = sum(w in text for w in words)
        if score:
            scored.append((score, {"id": pid, "title": p["title"], "year": p["year"]}))
    scored.sort(key=lambda s: -s[0])
    return json.dumps([paper for _, paper in scored[:max_results]])

In [ ]:
%%writefile -a paper_server.py

@mcp.tool()
def cite_paper(paper_id: str) -> str:
    """Format a short citation for one paper. Use this when you already know the paper's arXiv id,
    e.g. "2310.06625". Fails on an unknown id; use search_papers first."""
    if paper_id not in PAPERS:
        raise ValueError(f"Unknown paper id {paper_id!r}. Use search_papers to find an id.")
    p = PAPERS[paper_id]
    return f'{p["authors"]} ({p["year"]}). {p["title"]}. arXiv:{paper_id}'

In [ ]:
%%writefile -a paper_server.py

@mcp.resource("paper://{paper_id}")
def get_paper(paper_id: str) -> str:
    """One paper's full record, as JSON."""
    return json.dumps({"id": paper_id, **PAPERS[paper_id]})

@mcp.prompt()
def summarize_paper(paper_id: str) -> str:
    """Ask for a three-bullet summary of one paper, for a report."""
    return f"Read paper://{paper_id} and summarize it in three short bullets for a report on transformers for time series."

if __name__ == "__main__":
    mcp.run()  # stdio by default

## 2. Connect a client over stdio

The connection lifecycle (§21.2.3): **start the server** → **initialize** (version and capabilities) → **operate** → **shut down**

In a notebook we keep the session across many cells, so the session lives in its own task
(open it with `async with` in one cell and close it in another, and asyncio raises an error)

In [ ]:
import os, sys, json, asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
async def run_session(params, ready, stop):
    with open("server.log", "w") as log:
        async with stdio_client(params, errlog=log) as (read, write):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                ready.set_result((session, init))
                await stop.wait()

async def connect(server_file):
    ready, stop = asyncio.get_running_loop().create_future(), asyncio.Event()
    params = StdioServerParameters(command=sys.executable, args=[server_file])
    task = asyncio.create_task(run_session(params, ready, stop))
    session, init = await ready

    async def close():
        stop.set()
        await task
    return session, init, close

In [ ]:
session, init, close = await connect("paper_server.py")
print("server:", init.serverInfo.name, init.serverInfo.version, "· protocol", init.protocolVersion)
caps = init.capabilities
print("tools:", caps.tools is not None, "· resources:", caps.resources is not None,
      "· prompts:", caps.prompts is not None, "· sampling: not offered by servers")

## 3. Ask the server what it offers

`tools/list` · `resources/templates/list` · `prompts/list` — the client needn't know anything about the server in advance

In [ ]:
listed = await session.list_tools()
for t in listed.tools:
    print(f"🔧 {t.name}\n   {t.description}\n   inputSchema: {json.dumps(t.inputSchema)}\n")

In [ ]:
templates = await session.list_resource_templates()
for r in templates.resourceTemplates:
    print("📄", r.uriTemplate, "-", r.description)
prompts = await session.list_prompts()
for p in prompts.prompts:
    print("📝", p.name, [a.name for a in p.arguments], "-", p.description)

## 4. Call a tool · read a resource · fetch a prompt

In [ ]:
result = await session.call_tool("search_papers", {"query": "transformer time series", "max_results": 3})
print("isError:", result.isError)
print(result.content[0].text)

In [ ]:
res = await session.read_resource("paper://2310.06625")
print(res.contents[0].uri, "\n", res.contents[0].text)

pr = await session.get_prompt("summarize_paper", {"paper_id": "2310.06625"})
print(pr.messages[0].role, ":", pr.messages[0].content.text)

In [ ]:
bad = await session.call_tool("cite_paper", {"paper_id": "9999.99999"})
print("isError:", bad.isError)
print(bad.content[0].text)

## 5. Let the model pick a tool from the server's descriptions

**Model:** `google/gemma-4-26b-a4b-it` — the same model as every episode, pinned to Makora
· this lab needs **tool calling** (the model returns a tool name + arguments as JSON), which Gemma 4 can do
· USD 0.09 per million input tokens and USD 0.28 per million output tokens — one tool choice costs about 0.002 cents
· The `:free` version doesn't work (tested 2026-11-25: 3 out of 4 calls got 429 "rate-limited")

MCP gives the tool's **schema**; the model's API wants tools in **its own format** — the book notes the argument JSON schema is the same (§21.9.2),
so the conversion takes a few lines

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}

In [ ]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0}

In [ ]:
def to_openai(tool):
    """MCP tool → the tool format of the OpenAI-style chat API."""
    return {"type": "function",
            "function": {"name": tool.name, "description": tool.description,
                         "parameters": tool.inputSchema}}

TOOLS = [to_openai(t) for t in listed.tools]

In [ ]:
async def pick_and_call(question):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA, tools=TOOLS,
                                       messages=[{"role": "user", "content": question}])
    COST["chat"] += r.usage.cost
    call = r.choices[0].message.tool_calls[0]
    args = json.loads(call.function.arguments)
    print(f"❓ {question}\n🤖 model picked: {call.function.name}({args})")
    result = await session.call_tool(call.function.name, args)
    print("📦", result.content[0].text, "\n")

In [ ]:
await pick_and_call("Find papers about transformers for time series forecasting.")
await pick_and_call("Give me a citation for paper 2310.06625.")

In [ ]:
await close()
print(f"cost: ${COST['chat']:.6f}")

## Summary

| Step | What we saw |
|---|---|
| server | FastMCP + three decorators: `@mcp.tool()` · `@mcp.resource("paper://{paper_id}")` · `@mcp.prompt()` |
| initialize | the server declares tools · resources · prompts |
| `tools/list` | inputSchema built from type hints, the description is the docstring — no hand-written JSON schema |
| a bad `tools/call` | no exception — `isError: True` + a message the model can read |
| the model picks a tool | find by topic → `search_papers` · known id → `cite_paper` |

**Exercises**
1. Change `cite_paper`'s docstring to just `"Cite."` and ask the second question again — does the model still pick right?
2. Add a tool `list_years()` that returns every year in the library, and see it appear in `tools/list` without changing the client
3. Open `server.log` and see which requests the server received

**Next — Lab 2:** look inside the line: we type JSON-RPC messages to this same server by hand, from `initialize` to an error